In [ ]:
pip install ollama

In [ ]:
pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
from ollama import Client

In [ ]:
class MultiLLM:
    def __init__(self):
        self.client = Client(
            host="https://ollama.com",
            headers={"Authorization": "Bearer " + os.getenv("OLLAMA_API_KEY")}
        )

        self.models = {
            "orchestrator": "gpt-oss:120b",
            "responder": "gpt-oss:120b",
            "critic": "gpt-oss:120b",
            "judge": "gpt-oss:120b"
        }

    def _call(self, role, system_prompt, user_prompt):
        response = self.client.chat(
            model=self.models[role],
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return response["message"]["content"]

    def responder(self, system_prompt, user_prompt):
        return self._call("responder", system_prompt, user_prompt)

    def critic(self, system_prompt, user_prompt):
        return self._call("critic", system_prompt, user_prompt)

    def judge(self, system_prompt, user_prompt):
        return self._call("judge", system_prompt, user_prompt)

    def orchestrator(self, system_prompt, user_prompt):
        return self._call("orchestrator", system_prompt, user_prompt)

In [ ]:
responder_prompt = '''
You generate, defend, and improve answers.

Inputs:
- user_query
- current_answer
- latest_critique

Rules:
- If no current_answer → generate answer from user_query
- If critique exists → improve answer but also defend strong points
- Do not blindly accept critique
- Be confident but open-minded
- Refine, expand, and justify where needed
- Avoid unnecessary length

Output format (strict JSON):
{
  "answer": "refined answer",
  "stance": "DEFENDED | MODIFIED"
}
'''

In [ ]:
orchestrator_prompt = '''
You are the orchestrator of a multi-agent LLM debate system.

Your job: decide whether the debate should CONTINUE or STOP.

Inputs:
- user_query: original question
- current_answer: latest improved answer
- latest_critique: most recent critique
- judge_feedback: judge's evaluation

Decision rules:
- CONTINUE if any flaw, gap, inconsistency, or weak reasoning exists
- STOP only if answer is complete, correct, clear, and no major issues remain
- Be strict and avoid unnecessary iterations


Output format (strict JSON):
{
  "decision": "CONTINUE"
}
OR
{
  "decision": "STOP",
  "reason": "max 10 words explanation"
}
'''

In [ ]:
critic_prompt = '''
You are a strict critic.

Inputs:
- user_query
- current_answer

Rules:
- Point out flaws, gaps, weak reasoning, missing details
- Be direct and argumentative
- Do not rewrite full answer
- If answer is mostly correct, acknowledge briefly but still test it
- If no major issue → say it is acceptable but mention minor improvement if any
- Avoid repeating same points

Output format (strict JSON):
{
  "critique": "direct critique text",
  "severity": "LOW | MEDIUM | HIGH"
}
'''

In [ ]:
judge_prompt = '''
You evaluate answer quality.

Inputs:
- user_query
- current_answer
- latest_critique

Rules:
- Check correctness, completeness, clarity
- Be strict

Output format (strict JSON):
{
  "verdict": "ACCEPT | REJECT",
  "confidence": 0-1,
  "reason": "max 10 words"
}
'''

In [ ]:
llm = MultiLLM()

In [ ]:
current_answer = ""
latest_critique = ""
judge_feedback = ""

In [ ]:
user_prompt = input("Enter your Prompt: ")

Enter your Prompt: Hello


In [ ]:
import json
import re

def extract_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError("No valid JSON found")

In [ ]:
def run_responder():
    global current_answer

    responder_input = f"""
    user_query: {user_prompt}
    current_answer: {current_answer}
    latest_critique: {latest_critique}
    """

    response = llm.responder(responder_prompt, responder_input)

    try:
        data = extract_json(response)
        current_answer = data.get("answer", current_answer)
        print("[ANSWER]:", current_answer)
    except:
        print("[ERROR] Responder failed")

In [ ]:
def run_critic():
    global latest_critique

    critic_input = f"""
    user_query: {user_prompt}
    current_answer: {current_answer}
    """

    response = llm.critic(critic_prompt, critic_input)

    try:
        data = extract_json(response)
        latest_critique = data.get("critique", "")
        print("[CRITIQUE]:", latest_critique)
    except:
        print("[ERROR] Critic failed")

In [ ]:
def run_improve():
    global current_answer

    responder_input = f"""
    user_query: {user_prompt}
    current_answer: {current_answer}
    latest_critique: {latest_critique}
    """

    response = llm.responder(responder_prompt, responder_input)

    try:
        data = extract_json(response)
        current_answer = data.get("answer", current_answer)
        print("[IMPROVED]:", current_answer)
    except:
        print("[ERROR] Improve failed")

In [ ]:
def run_judge():
    global judge_feedback

    judge_input = f"""
    user_query: {user_prompt}
    current_answer: {current_answer}
    latest_critique: {latest_critique}
    """

    response = llm.judge(judge_prompt, judge_input)

    try:
        data = extract_json(response)
        judge_feedback = data.get("verdict", "REJECT")
        print("[JUDGE]:", judge_feedback)
    except:
        print("[ERROR] Judge failed")

In [ ]:
def run_orchestrator():
    orchestrator_input = f"""
    user_query: {user_prompt}
    current_answer: {current_answer}
    latest_critique: {latest_critique}
    judge_feedback: {judge_feedback}
    """

    response = llm.orchestrator(orchestrator_prompt, orchestrator_input)

    try:
        data = extract_json(response)
        decision = data.get("decision", "STOP")
        reason = data.get("reason", "")

        print("[DECISION]:", decision)

        return decision

    except:
        print("[ERROR] Orchestrator failed")
        return "STOP"

In [ ]:
max_iters = 5

for i in range(max_iters):
    print(f"\n======== ITERATION {i+1} ========")

    run_responder()
    run_critic()
    run_improve()
    run_judge()

    decision = run_orchestrator()

    if decision == "STOP":
        print("Stopping loop")
        break


======== ITERATION 1 ========
[ANSWER]: Hello! How can I assist you today?
[CRITIQUE]: The reply is essentially correct—it's a polite greeting and an offer to help. However, it is overly generic and assumes the user needs assistance before confirming their intent, which might feel presumptuous. A slightly more tailored or open-ended follow‑up would improve the interaction.
[IMPROVED]: Hello! How are you today? Let me know if there's anything you'd like to discuss or any question I can help with.
[JUDGE]: ACCEPT
[DECISION]: STOP
Stopping loop
